# Machine Learning

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [6]:
df = pd.read_csv('../data/HKHJ_Dataset_After_MV.csv')

In [7]:
df.shape

(158563, 82)

Columns that can be dropped:
- info_rating
- info_class
- location
- going
- horse_name_id
- jockey
- trainer
- surface_type

In [8]:
df

,date,location,going,horse_number,horse_name_id,jockey,trainer,weight,on_date_horse_weight,draw,...,weight_trend_missing,weight_trend_direction_missing,days_since_last_race_any_missing,days_since_last_race_same_season_missing,info_rating_high_missing,info_rating_low_missing,info_rating_mid_missing,on_date_horse_weight_missing,draw_missing,horse_number_missing
0,2009-01-01 00:00:00+08:00,Sha Tin,GOOD TO FIRM,1.0,CH130,K C Leung,Y S Tsui,119,1114.0,4.0,...,1,1,1,1,0,0,0,0,0,0
1,2009-01-01 00:00:00+08:00,Sha Tin,GOOD TO FIRM,5.0,CJ125,W C Marwing,J Moore,126,1106.0,11.0,...,1,1,1,1,0,0,0,0,0,0
2,2009-01-01 00:00:00+08:00,Sha Tin,GOOD TO FIRM,6.0,CH066,E Saint-Martin,J Size,124,1223.0,3.0,...,1,1,1,1,0,0,0,0,0,0
3,2009-01-01 00:00:00+08:00,Sha Tin,GOOD TO FIRM,11.0,CE325,M L Yeung,A Lee,109,1213.0,9.0,...,1,1,1,1,0,0,0,0,0,0
4,2009-01-01 00:00:00+08:00,Sha Tin,GOOD TO FIRM,4.0,CJ197,D Whyte,C S Shum,126,1219.0,8.0,...,1,1,1,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158558,2025-07-13 00:00:00+08:00,Sha Tin,GOOD TO FIRM,14.0,J376,B Thompson,P F Yiu,116,1026.0,8.0,...,0,0,0,0,0,0,0,0,0,0
158559,2025-07-13 00:00:00+08:00,Sha Tin,GOOD TO FIRM,4.0,J210,Z Purton,M Newnham,131,1142.0,11.0,...,0,0,0,0,0,0,0,0,0,0
158560,2025-07-13 00:00:00+08:00,Sha Tin,GOOD TO FIRM,8.0,J550,K Teetan,J Size,124,1093.0,9.0,...,0,0,0,0,0,0,0,0,0,0
158561,2025-07-13 00:00:00+08:00,Sha Tin,GOOD TO FIRM,7.0,K198,K C Leung,D A Hayes,125,1115.0,10.0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
df.dtypes

date                             object
location                         object
going                            object
horse_number                    float64
horse_name_id                    object
                                 ...   
info_rating_low_missing           int64
info_rating_mid_missing           int64
on_date_horse_weight_missing      int64
draw_missing                      int64
horse_number_missing              int64
Length: 82, dtype: object

In [10]:
drop_columns = ['info_rating', 'info_class', 'location', 'going', 'horse_name_id', 'jockey', 'trainer', 'surface_type']
df_cleaned = df.drop(columns=drop_columns)

In [11]:
df_cleaned

,date,horse_number,weight,on_date_horse_weight,draw,info_distance,rail_offset,unique_id,season,races_in_season,...,weight_trend_missing,weight_trend_direction_missing,days_since_last_race_any_missing,days_since_last_race_same_season_missing,info_rating_high_missing,info_rating_low_missing,info_rating_mid_missing,on_date_horse_weight_missing,draw_missing,horse_number_missing
0,2009-01-01 00:00:00+08:00,1.0,119,1114.0,4.0,1600,2,1.0,1,1,...,1,1,1,1,0,0,0,0,0,0
1,2009-01-01 00:00:00+08:00,5.0,126,1106.0,11.0,1000,2,8.0,1,1,...,1,1,1,1,0,0,0,0,0,0
2,2009-01-01 00:00:00+08:00,6.0,124,1223.0,3.0,1000,2,8.0,1,1,...,1,1,1,1,0,0,0,0,0,0
3,2009-01-01 00:00:00+08:00,11.0,109,1213.0,9.0,1000,2,8.0,1,1,...,1,1,1,1,0,0,0,0,0,0
4,2009-01-01 00:00:00+08:00,4.0,126,1219.0,8.0,1000,2,8.0,1,1,...,1,1,1,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158558,2025-07-13 00:00:00+08:00,14.0,116,1026.0,8.0,1400,0,12627.0,16,8,...,0,0,0,0,0,0,0,0,0,0
158559,2025-07-13 00:00:00+08:00,4.0,131,1142.0,11.0,1400,0,12627.0,16,8,...,0,0,0,0,0,0,0,0,0,0
158560,2025-07-13 00:00:00+08:00,8.0,124,1093.0,9.0,1400,0,12627.0,16,8,...,0,0,0,0,0,0,0,0,0,0
158561,2025-07-13 00:00:00+08:00,7.0,125,1115.0,10.0,1400,0,12627.0,16,11,...,0,0,0,0,0,0,0,0,0,0


## First Model: LightGBM

In [79]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import classification_report


In [80]:
# 1. Sortieren und Zeit-Splits definieren
gb_df = df_cleaned.sort_values('date').copy()
train = gb_df[gb_df['season'] <= 14]
valid = gb_df[gb_df['season'] == 15]
test  = gb_df[gb_df['season'] == 16]

In [81]:
drop_cols = ['unique_id', 'date', 
             'win'] #top3, win

In [82]:

"""feature_cols = [c for c in gb_df.columns if c not in drop_cols + ['win']]
X_train, y_train = train[feature_cols], train['win']
X_valid, y_valid = valid[feature_cols], valid['win']
X_test,  y_test  = test[feature_cols],  test['win']"""


feature_cols = [c for c in gb_df.columns if c not in drop_cols + ['top3']]
X_train, y_train = train[feature_cols], train['top3']
X_valid, y_valid = valid[feature_cols], valid['top3']
X_test,  y_test  = test[feature_cols],  test['top3']

In [83]:
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
clf = lgb.LGBMClassifier(
    objective='binary',
    n_estimators=1500,
    learning_rate=0.03,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=3.0,
    class_weight=None,
    scale_pos_weight=pos_weight
)

In [84]:
clf.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(200)]
)

[LightGBM] [Info] Number of positive: 32950, number of negative: 105233
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005283 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5799
[LightGBM] [Info] Number of data points in the train set: 138183, number of used features: 67
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.238452 -> initscore=-1.161186
[LightGBM] [Info] Start training from score -1.161186
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[12]	valid_0's auc: 0.714869	valid_0's binary_logloss: 0.539354


,boosting_type,'gbdt'
,num_leaves,64
,max_depth,-1
,learning_rate,0.03
,n_estimators,1500
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [85]:
# 4. Baseline-Metriken
def report_metrics(split_name, y_true, y_pred):
    print(f'{split_name} ROC-AUC: {roc_auc_score(y_true, y_pred):.3f}')
    print(f'{split_name} PR-AUC:  {average_precision_score(y_true, y_pred):.3f}')

In [86]:
report_metrics('Train', y_train, clf.predict_proba(X_train)[:, 1])
report_metrics('Valid', y_valid, clf.predict_proba(X_valid)[:, 1])
report_metrics('Test',  y_test,  clf.predict_proba(X_test)[:, 1])

Train ROC-AUC: 0.725
Train PR-AUC:  0.441
Valid ROC-AUC: 0.715
Valid PR-AUC:  0.443
Test ROC-AUC: 0.711
Test PR-AUC:  0.419


In [88]:
threshold = 0.35  # Beispiel: wähle einen sinnvollen Cutoff z.B. aus dem PR-Knick
y_pred = (clf.predict_proba(X_valid)[:, 1] >= threshold).astype(int)

print(classification_report(y_valid, y_pred, target_names=['not_top3', 'top3']))

              precision    recall  f1-score   support

    not_top3       0.82      0.83      0.82      7586
        top3       0.45      0.44      0.45      2496

    accuracy                           0.73     10082
   macro avg       0.64      0.63      0.63     10082
weighted avg       0.73      0.73      0.73     10082

